# Задание III. Кластеризация `baseball.csv` — вариант 7, готовый ноутбук для сдачи преподавателю

Этот ноутбук реализует **шаги 1–10** задания по кластеризации на наборе `baseball.csv`.

Я сделал его максимально учебным:
- почти перед каждым блоком есть объяснение, **зачем** он нужен;
- код разбит на небольшие логические части;
- важные решения отдельно прокомментированы;
- весь анализ собран в класс, как просит пункт 8 задания.

## Что именно делает ноутбук

1. Загружает данные и проводит первичный осмотр.
2. Обрабатывает пропуски в `Salary` выбранным методом.
3. Пересчитывает `logSalary = log(1 + Salary)`.
4. Нормализует числовые признаки и кодирует категориальные через `OneHotEncoder`.
5. Строит агломеративную иерархическую кластеризацию и дендрограмму.
6. Считает критерий pseudo-F для числа кластеров от 2 до 20.
7. Выбирает оптимальное число кластеров как **первый локальный пик**.
8. Строит 2D-проекцию по методу из варианта.
9. Выполняет кластеризацию методом сферических кластеров с прототипом.
10. Находит наиболее типичных представителей каждого кластера.
11. Повторяет весь цикл после дополнительного логарифмирования асимметричных признаков.
12. Повторяет цикл после отбора признаков методом **VarClus-подобной** кластеризации переменных.

## Важное замечание про вариант

В файле задания методы зависят от номера варианта.  
Этот файл подготовлен именно для **варианта 7** и уже настроен под сдачу.

В параметрах уже зафиксировано:

```python
ACTIVE_VARIANT = 7
```

Я выбрал `6` как рабочее значение по умолчанию, потому что в имени файла задания есть `Task2_26.pdf`, и это можно трактовать как вариант с последней цифрой `6`.  
Если у тебя другой вариант, просто измени **одну цифру** в начале ноутбука — весь остальной код подстроится автоматически.

## Почему шаги 11–12 здесь не реализованы

В самом тексте задания сказано, что для проверки нужно прислать **JN, реализующий шаги 1–10**.  
Поэтому этот ноутбук сфокусирован именно на кластеризации по `baseball.csv`.

## 0. Импорт библиотек

Ниже подключаются стандартные библиотеки для анализа данных, визуализации и машинного обучения.

### Что используется и зачем
- `pandas`, `numpy` — таблицы и массивы.
- `matplotlib` — графики.
- `scipy` — иерархическая кластеризация.
- `scikit-learn` — импутация, масштабирование, кодирование, критерий pseudo-F, PCA, t-SNE, NMF, KMeans, EM.
- `torch` — только для варианта с **AE (autoencoder)**.  
  Если `torch` не установлен, ноутбук автоматически сделает безопасный запасной вариант через PCA и честно сообщит об этом.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA, NMF
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.manifold import TSNE
from sklearn.metrics import calinski_harabasz_score, pairwise_distances
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    MinMaxScaler,
    MaxAbsScaler,
    Normalizer,
    OneHotEncoder,
)
from sklearn.cluster import KMeans

## 1. Параметры варианта

Сначала задаём активный вариант.  
Ниже есть словарь `VARIANT_CONFIGS`, где собраны все методы из таблицы задания.

### Как это устроено
Для каждого варианта хранятся:
- способ обработки пропусков;
- способ масштабирования;
- параметры иерархической кластеризации;
- метод проекции на плоскость;
- метод сферической кластеризации с прототипом;
- число групп для шага с VarClus.

In [ ]:
ACTIVE_VARIANT = 7   # вариант 7 уже зафиксирован под сдачу

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

VARIANT_CONFIGS = {
    1: {
        "imputer": "median",
        "scaler": "standard",
        "linkage": "ward",
        "distance": "euclidean",
        "projection": "ae",
        "prototype_method": "kmeans",
        "varclus_clusters": 7,
    },
    2: {
        "imputer": "mean",
        "scaler": "robust",
        "linkage": "complete",
        "distance": "euclidean",
        "projection": "tsne",
        "prototype_method": "kmedoids",
        "varclus_clusters": 5,
    },
    3: {
        "imputer": "knn3",
        "scaler": "minmax",
        "linkage": "complete",
        "distance": "manhattan",
        "projection": "som",
        "prototype_method": "em",
        "varclus_clusters": 7,
    },
    4: {
        "imputer": "knn5",
        "scaler": "maxabs",
        "linkage": "average",
        "distance": "euclidean",
        "projection": "pca",
        "prototype_method": "kmeans",
        "varclus_clusters": 5,
    },
    5: {
        "imputer": "knn7",
        "scaler": "normalizer",
        "linkage": "average",
        "distance": "manhattan",
        "projection": "nmf",
        "prototype_method": "kmedoids",
        "varclus_clusters": 7,
    },
    6: {
        "imputer": "median",
        "scaler": "standard",
        "linkage": "ward",
        "distance": "euclidean",
        "projection": "ae",
        "prototype_method": "em",
        "varclus_clusters": 5,
    },
    7: {
        "imputer": "mean",
        "scaler": "robust",
        "linkage": "complete",
        "distance": "euclidean",
        "projection": "tsne",
        "prototype_method": "kmeans",
        "varclus_clusters": 7,
    },
    8: {
        "imputer": "knn3",
        "scaler": "minmax",
        "linkage": "complete",
        "distance": "manhattan",
        "projection": "som",
        "prototype_method": "kmedoids",
        "varclus_clusters": 5,
    },
    9: {
        "imputer": "knn5",
        "scaler": "maxabs",
        "linkage": "average",
        "distance": "euclidean",
        "projection": "pca",
        "prototype_method": "em",
        "varclus_clusters": 7,
    },
    0: {
        "imputer": "knn7",
        "scaler": "normalizer",
        "linkage": "average",
        "distance": "manhattan",
        "projection": "nmf",
        "prototype_method": "kmeans",
        "varclus_clusters": 5,
    },
}

config = VARIANT_CONFIGS[ACTIVE_VARIANT]
print("Активный вариант:", ACTIVE_VARIANT)
print(pd.Series(config))

## 2. Загрузка данных и первичный осмотр

Здесь мы просто открываем `baseball.csv` и смотрим:
- размер таблицы;
- список признаков;
- типы данных;
- сколько пропусков в каждом столбце.

### Почему это обязательно
Перед любым анализом нужно понять структуру данных.  
Особенно важно сразу увидеть:
- где находятся пропуски;
- какие признаки числовые, а какие категориальные;
- какой столбец является идентификатором.

По условию задания `Name` — это **ID записи**, а не обычный признак.

In [ ]:
DATA_PATH = "baseball.csv"

df_raw = pd.read_csv(DATA_PATH)
print("Размер таблицы:", df_raw.shape)
display(df_raw.head())
display(df_raw.info())
display(df_raw.isna().sum().sort_values(ascending=False))

### Небольшой вывод по осмотру данных

Обычно здесь видно следующее:
- `Name` — идентификатор игрока;
- `Salary` и `logSalary` содержат пропуски;
- часть признаков числовые, часть категориальные;
- `logSalary` в исходном файле есть, но по заданию его надо **пересчитать заново** из `Salary`.

Это важный момент: мы не доверяем старому столбцу `logSalary`, а строим его сами как:

\[
\logSalary = \log(1 + Salary)
\]

Так мы получаем более симметричное распределение и одновременно выполняем требование задания.

## 3. Вспомогательные функции

Ниже идут служебные функции и маленькие алгоритмы, которые понадобятся дальше.

### Что здесь будет
1. Выбор колонок.
2. Построение импьютера и масштабировщика.
3. Поиск признаков с тяжёлым правым хвостом.
4. Свой K-Medoids.
5. Небольшая реализация SOM.
6. Проекция через AE.
7. Выбор оптимального числа кластеров по первому локальному пику pseudo-F.
8. Подбор наиболее типичного представителя кластера.

In [ ]:
def get_column_groups(df: pd.DataFrame):
    id_col = "Name"
    categorical_cols = ["Team", "League", "Division", "Position", "Div"]
    numeric_cols = [c for c in df.columns if c not in categorical_cols + [id_col, "logSalary"]]
    return id_col, numeric_cols, categorical_cols


def build_imputer(kind: str):
    if kind == "median":
        return SimpleImputer(strategy="median")
    if kind == "mean":
        return SimpleImputer(strategy="mean")
    if kind == "knn3":
        return KNNImputer(n_neighbors=3)
    if kind == "knn5":
        return KNNImputer(n_neighbors=5)
    if kind == "knn7":
        return KNNImputer(n_neighbors=7)
    raise ValueError(f"Неизвестный тип импьютера: {kind}")


def build_scaler(kind: str):
    if kind == "standard":
        return StandardScaler()
    if kind == "robust":
        return RobustScaler()
    if kind == "minmax":
        return MinMaxScaler()
    if kind == "maxabs":
        return MaxAbsScaler()
    if kind == "normalizer":
        return Normalizer()
    raise ValueError(f"Неизвестный тип масштабирования: {kind}")


def first_local_peak(xs: List[int], ys: List[float]) -> Tuple[int, int]:
    """
    Возвращает:
    - значение x первого локального максимума;
    - индекс этого максимума в списке.

    Если локального максимума нет, берём глобальный максимум.
    """
    values = np.asarray(ys, dtype=float)
    for i in range(1, len(values) - 1):
        if values[i] > values[i - 1] and values[i] >= values[i + 1]:
            return xs[i], i
    idx = int(np.nanargmax(values))
    return xs[idx], idx


def find_right_skewed_columns(
    numeric_df: pd.DataFrame,
    skew_threshold: float = 1.0,
    exclude: Optional[List[str]] = None
) -> List[str]:
    """
    Ищем признаки:
    - с одной модой;
    - с тяжёлым правым хвостом (skew > skew_threshold);
    - без отрицательных значений.

    Исключаем Salary и logSalary, потому что логарифм зарплаты уже создаётся
    отдельно по условию задания и второй раз логарифмировать его не нужно.
    """
    exclude = exclude or []
    selected = []
    for col in numeric_df.columns:
        if col in exclude:
            continue
        s = numeric_df[col].dropna()
        if len(s) < 3:
            continue
        one_mode = s.mode(dropna=True).shape[0] == 1
        heavy_right_tail = s.skew() > skew_threshold
        non_negative = s.min() >= 0
        if one_mode and heavy_right_tail and non_negative:
            selected.append(col)
    return selected


def fit_kmedoids(
    X: np.ndarray,
    n_clusters: int,
    metric: str = "euclidean",
    random_state: int = 42,
    max_iter: int = 100,
):
    """
    Простая реализация PAM/K-Medoids без внешних зависимостей.
    Нужна для вариантов, где по условию требуется KMedoids.
    """
    rng = np.random.default_rng(random_state)
    metric_name = "cityblock" if metric == "manhattan" else metric
    D = pairwise_distances(X, metric=metric_name)

    n_samples = X.shape[0]
    medoid_indices = rng.choice(n_samples, size=n_clusters, replace=False)

    for _ in range(max_iter):
        labels = np.argmin(D[:, medoid_indices], axis=1)
        new_medoids = medoid_indices.copy()
        changed = False

        for k in range(n_clusters):
            members = np.where(labels == k)[0]

            if len(members) == 0:
                candidates = [i for i in range(n_samples) if i not in new_medoids]
                new_medoids[k] = rng.choice(candidates)
                changed = True
                continue

            local_dist_sum = D[np.ix_(members, members)].sum(axis=1)
            best_medoid = members[np.argmin(local_dist_sum)]

            if best_medoid != medoid_indices[k]:
                new_medoids[k] = best_medoid
                changed = True

        medoid_indices = new_medoids
        if not changed:
            break

    labels = np.argmin(D[:, medoid_indices], axis=1)

    return {
        "labels": labels,
        "medoid_indices": medoid_indices,
        "distance_matrix": D,
    }


def som_projection(
    X: np.ndarray,
    grid: Tuple[int, int] = (10, 10),
    n_iter: int = 1000,
    learning_rate: float = 0.5,
    sigma: Optional[float] = None,
    random_state: int = 42,
):
    """
    Очень компактная реализация SOM, достаточная для учебного задания.

    Идея:
    - есть двумерная решётка нейронов;
    - каждый нейрон имеет вектор весов той же размерности, что и объект;
    - на каждом шаге ищется лучший нейрон (BMU);
    - затем он и его соседи сдвигаются к текущему объекту.

    В конце каждая строка данных отображается в координаты BMU на решётке.
    """
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, dtype=float)

    n_samples, n_features = X.shape
    gx, gy = grid
    if sigma is None:
        sigma = max(gx, gy) / 2

    # Инициализация весов случайными объектами из выборки
    init_idx = rng.choice(n_samples, size=gx * gy, replace=True)
    weights = X[init_idx].reshape(gx, gy, n_features).copy()

    grid_coords = np.array([(i, j) for i in range(gx) for j in range(gy)])

    for t in range(n_iter):
        x = X[rng.integers(0, n_samples)]

        distances = np.linalg.norm(weights - x, axis=2)
        bmu = np.unravel_index(np.argmin(distances), distances.shape)
        bmu_coord = np.array(bmu)

        lr_t = learning_rate * np.exp(-t / n_iter)
        sigma_t = sigma * np.exp(-t / n_iter)

        grid_dist = np.sum((grid_coords - bmu_coord) ** 2, axis=1).reshape(gx, gy)
        neighborhood = np.exp(-grid_dist / (2 * sigma_t ** 2))

        weights += lr_t * neighborhood[..., None] * (x - weights)

    coords = np.zeros((n_samples, 2), dtype=float)
    for i, x in enumerate(X):
        distances = np.linalg.norm(weights - x, axis=2)
        bmu = np.unravel_index(np.argmin(distances), distances.shape)
        coords[i] = bmu

    # Небольшой шум, чтобы точки меньше наслаивались друг на друга на графике
    coords += rng.normal(scale=0.08, size=coords.shape)

    return coords, weights


def ae_projection(X: np.ndarray, epochs: int = 120, random_state: int = 42):
    """
    2D-проекция через простой автокодировщик.

    Если torch недоступен, делаем честный fallback на PCA,
    чтобы ноутбук не ломался.
    """
    try:
        import torch
        import torch.nn as nn

        torch.manual_seed(random_state)
        np.random.seed(random_state)

        X_np = np.asarray(X, dtype=np.float32)
        X_tensor = torch.tensor(X_np)

        input_dim = X_np.shape[1]
        hidden_dim = min(32, max(8, input_dim // 2))
        batch_size = 64

        class AutoEncoder(nn.Module):
            def __init__(self, input_dim, hidden_dim):
                super().__init__()
                self.encoder = nn.Sequential(
                    nn.Linear(input_dim, hidden_dim),
                    nn.ReLU(),
                    nn.Linear(hidden_dim, 2),
                )
                self.decoder = nn.Sequential(
                    nn.Linear(2, hidden_dim),
                    nn.ReLU(),
                    nn.Linear(hidden_dim, input_dim),
                )

            def forward(self, x):
                z = self.encoder(x)
                x_hat = self.decoder(z)
                return x_hat, z

        model = AutoEncoder(input_dim, hidden_dim)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
        loss_fn = nn.MSELoss()

        for _ in range(epochs):
            perm = torch.randperm(X_tensor.size(0))
            for start in range(0, X_tensor.size(0), batch_size):
                batch_idx = perm[start:start + batch_size]
                batch = X_tensor[batch_idx]

                optimizer.zero_grad()
                reconstructed, _ = model(batch)
                loss = loss_fn(reconstructed, batch)
                loss.backward()
                optimizer.step()

        with torch.no_grad():
            _, latent = model(X_tensor)

        return latent.numpy(), "AE (torch)"

    except Exception as e:
        print("Предупреждение: AE через torch недоступен, переключаюсь на PCA.")
        print("Причина:", e)
        coords = PCA(n_components=2).fit_transform(X)
        return coords, "PCA fallback"


def make_projection(X: np.ndarray, method: str, random_state: int = 42):
    method = method.lower()

    if method == "pca":
        coords = PCA(n_components=2).fit_transform(X)
        return coords, "PCA"

    if method == "tsne":
        perplexity = min(30, max(5, X.shape[0] // 10))
        coords = TSNE(
            n_components=2,
            random_state=random_state,
            init="pca",
            learning_rate="auto",
            perplexity=perplexity,
        ).fit_transform(X)
        return coords, "t-SNE"

    if method == "nmf":
        X_nonneg = np.maximum(X, 0)
        coords = NMF(
            n_components=2,
            init="nndsvda",
            random_state=random_state,
            max_iter=1000,
        ).fit_transform(X_nonneg)
        return coords, "NMF"

    if method == "som":
        coords, _ = som_projection(X, random_state=random_state)
        return coords, "SOM"

    if method == "ae":
        return ae_projection(X, random_state=random_state)

    raise ValueError(f"Неизвестный метод проекции: {method}")


def make_hierarchical_linkage(X: np.ndarray, linkage_method: str, distance: str):
    """
    Для Ward допустимо только евклидово расстояние по матрице наблюдений.
    Для average/complete с Manhattan сначала считаем попарные расстояния.
    """
    if linkage_method == "ward":
        return linkage(X, method="ward", metric="euclidean")

    metric_name = "cityblock" if distance == "manhattan" else distance
    condensed = pdist(X, metric=metric_name)
    return linkage(condensed, method=linkage_method)


def prototype_clustering(
    X: np.ndarray,
    n_clusters: int,
    method: str,
    metric: str,
    random_state: int = 42,
):
    method = method.lower()

    if method == "kmeans":
        model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=20)
        labels = model.fit_predict(X)
        return {
            "labels": labels,
            "centers": model.cluster_centers_,
            "model": model,
            "method_label": "KMeans",
        }

    if method == "em":
        # Берём spherical covariance, чтобы соответствовать идее сферических кластеров
        model = GaussianMixture(
            n_components=n_clusters,
            covariance_type="spherical",
            random_state=random_state,
            n_init=10,
        )
        labels = model.fit_predict(X)
        return {
            "labels": labels,
            "centers": model.means_,
            "model": model,
            "method_label": "EM (GaussianMixture, spherical)",
        }

    if method == "kmedoids":
        result = fit_kmedoids(X, n_clusters=n_clusters, metric=metric, random_state=random_state)
        result["method_label"] = "KMedoids"
        return result

    raise ValueError(f"Неизвестный метод прототипной кластеризации: {method}")


def typical_representatives(
    X: np.ndarray,
    names: List[str],
    clustering_result: Dict,
    method: str,
):
    """
    Возвращает наиболее типичного представителя в каждом кластере.

    Логика:
    - KMeans / EM: берём объект, ближайший к центру.
    - KMedoids: сам медоид уже является типичным представителем.
    """
    X = np.asarray(X, dtype=float)
    labels = np.asarray(clustering_result["labels"])
    method = method.lower()

    rows = []
    for cluster_id in sorted(np.unique(labels)):
        idx = np.where(labels == cluster_id)[0]

        if method in {"kmeans", "em"}:
            center = clustering_result["centers"][cluster_id]
            distances = np.linalg.norm(X[idx] - center, axis=1)
            best_idx = idx[np.argmin(distances)]

        elif method == "kmedoids":
            best_idx = clustering_result["medoid_indices"][cluster_id]

        else:
            raise ValueError(f"Неизвестный метод: {method}")

        rows.append({
            "cluster": int(cluster_id),
            "row_index": int(best_idx),
            "Name": names[best_idx],
            "cluster_size": int((labels == cluster_id).sum()),
        })

    return pd.DataFrame(rows)


def varclus_like_selection(numeric_df: pd.DataFrame, n_clusters: int):
    """
    Практическая замена VarClus без внешних пакетов.

    Идея:
    1. Кластеризуем не объекты, а ПЕРЕМЕННЫЕ.
    2. Близость переменных оцениваем через |corr|.
    3. В каждом кластере признаков выбираем представителя:
       тот признак, который сильнее всего связан с остальными
       признаками своего кластера.

    Это не точная копия SAS VarClus, но для учебной работы
    логика очень близкая и интерпретируемая.
    """
    corr = numeric_df.corr(method="spearman").abs().fillna(0.0)
    variable_Z = linkage(pdist(numeric_df.T, metric="correlation"), method="average")
    labels = fcluster(variable_Z, t=n_clusters, criterion="maxclust")

    selected = []
    clusters = {}

    cols = list(numeric_df.columns)
    for lab in sorted(np.unique(labels)):
        members = [c for c, l in zip(cols, labels) if l == lab]
        clusters[int(lab)] = members

        if len(members) == 1:
            selected.append(members[0])
        else:
            subcorr = corr.loc[members, members]
            representative = subcorr.mean(axis=1).idxmax()
            selected.append(representative)

    return selected, clusters, variable_Z


def plot_projection(coords, labels, title: str):
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(coords[:, 0], coords[:, 1], c=labels, s=35, alpha=0.8)
    plt.title(title)
    plt.xlabel("Компонента / координата 1")
    plt.ylabel("Компонента / координата 2")
    plt.grid(alpha=0.3)
    plt.show()


def print_cluster_sizes(labels, title="Размеры кластеров"):
    s = pd.Series(labels).value_counts().sort_index()
    print(title)
    display(s.rename("count").to_frame())

## 4. Основной класс анализа

По условию задания шаги 3–7 надо оформить в виде **функции или класса**.  
Я сделал именно **класс**, потому что так удобнее:

- в одном месте лежат все параметры;
- результаты разных запусков удобно хранить вместе;
- проще повторять один и тот же конвейер для шагов 8, 9 и 10.

### Логика класса
Класс умеет:
1. подготовить данные;
2. выполнить иерархическую кластеризацию;
3. посчитать pseudo-F;
4. выбрать число кластеров;
5. построить проекцию;
6. выполнить прототипную кластеризацию;
7. найти типичных представителей.

In [ ]:
@dataclass
class RunResult:
    name: str
    X: np.ndarray
    feature_names: List[str]
    hier_labels: np.ndarray
    proto_labels: np.ndarray
    best_k: int
    pseudoF_table: pd.DataFrame
    projection_coords: np.ndarray
    projection_label: str
    representatives: pd.DataFrame
    skewed_columns: List[str]
    selected_numeric_features: Optional[List[str]] = None


class BaseballClusterStudy:
    def __init__(self, variant: int, random_state: int = 42):
        self.variant = variant
        self.config = VARIANT_CONFIGS[variant]
        self.random_state = random_state
        self.df_raw = None

    def load_data(self, path: str = "baseball.csv"):
        self.df_raw = pd.read_csv(path)
        return self.df_raw

    def _prepare_data(
        self,
        skew_transform: bool = False,
        selected_numeric_features: Optional[List[str]] = None,
    ):
        if self.df_raw is None:
            raise ValueError("Сначала вызови load_data().")

        df = self.df_raw.copy()
        id_col, numeric_cols, categorical_cols = get_column_groups(df)

        if selected_numeric_features is not None:
            numeric_cols = [c for c in numeric_cols if c in selected_numeric_features]
            if "Salary" not in numeric_cols:
                numeric_cols = numeric_cols + ["Salary"]

        imputer = build_imputer(self.config["imputer"])
        numeric_imputed = pd.DataFrame(
            imputer.fit_transform(df[numeric_cols]),
            columns=numeric_cols,
            index=df.index,
        )

        # logSalary пересчитываем строго по условию задания
        numeric_imputed["logSalary"] = np.log1p(numeric_imputed["Salary"])

        skewed_columns = []
        if skew_transform:
            skewed_columns = find_right_skewed_columns(
                numeric_imputed,
                skew_threshold=1.0,
                exclude=["Salary", "logSalary"],
            )
            for col in skewed_columns:
                numeric_imputed[col] = np.log1p(numeric_imputed[col])

            # logSalary после этого не меняем: он уже специально определён как log(1 + Salary)

        scaler = build_scaler(self.config["scaler"])
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

        all_numeric_cols = list(numeric_imputed.columns)

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", scaler, all_numeric_cols),
                ("cat", encoder, categorical_cols),
            ],
            remainder="drop",
        )

        model_input = pd.concat([numeric_imputed, df[categorical_cols]], axis=1)
        X = preprocessor.fit_transform(model_input)

        feature_names = list(preprocessor.get_feature_names_out())
        names = df[id_col].tolist()

        prepared = {
            "X": np.asarray(X, dtype=float),
            "feature_names": feature_names,
            "names": names,
            "numeric_imputed": numeric_imputed,
            "categorical_cols": categorical_cols,
            "skewed_columns": skewed_columns,
        }
        return prepared

    def _hierarchical_stage(self, X: np.ndarray):
        Z = make_hierarchical_linkage(
            X,
            linkage_method=self.config["linkage"],
            distance=self.config["distance"],
        )

        candidate_ks = list(range(2, 21))
        scores = []
        labels_by_k = {}

        for k in candidate_ks:
            labels = fcluster(Z, t=k, criterion="maxclust")
            labels_by_k[k] = labels

            if len(np.unique(labels)) < 2:
                score = np.nan
            else:
                score = calinski_harabasz_score(X, labels)
            scores.append(score)

        best_k, best_idx = first_local_peak(candidate_ks, scores)
        best_labels = labels_by_k[best_k]

        pseudoF_table = pd.DataFrame({
            "k": candidate_ks,
            "pseudoF": scores,
        })

        return {
            "linkage_matrix": Z,
            "candidate_ks": candidate_ks,
            "pseudoF_table": pseudoF_table,
            "best_k": best_k,
            "best_idx": best_idx,
            "labels": best_labels,
        }

    def run(
        self,
        run_name: str,
        skew_transform: bool = False,
        selected_numeric_features: Optional[List[str]] = None,
        show_plots: bool = True,
    ) -> RunResult:
        prepared = self._prepare_data(
            skew_transform=skew_transform,
            selected_numeric_features=selected_numeric_features,
        )

        X = prepared["X"]
        names = prepared["names"]

        hier = self._hierarchical_stage(X)
        best_k = hier["best_k"]
        hier_labels = hier["labels"]

        projection_coords, projection_label = make_projection(
            X,
            method=self.config["projection"],
            random_state=self.random_state,
        )

        proto = prototype_clustering(
            X,
            n_clusters=best_k,
            method=self.config["prototype_method"],
            metric=self.config["distance"],
            random_state=self.random_state,
        )
        proto_labels = proto["labels"]

        representatives = typical_representatives(
            X,
            names=names,
            clustering_result=proto,
            method=self.config["prototype_method"],
        )

        if show_plots:
            # Дендрограмма
            plt.figure(figsize=(12, 6))
            dendrogram(
                hier["linkage_matrix"],
                truncate_mode="lastp",
                p=20,
                leaf_rotation=90,
                leaf_font_size=10,
                show_contracted=True,
            )
            plt.title(f"{run_name}: дендрограмма (top 20 кластеров)")
            plt.xlabel("Склеенные группы / листья")
            plt.ylabel("Расстояние склейки")
            plt.grid(alpha=0.3)
            plt.show()

            # Pseudo-F
            plt.figure(figsize=(9, 5))
            plt.plot(hier["pseudoF_table"]["k"], hier["pseudoF_table"]["pseudoF"], marker="o")
            best_row = hier["pseudoF_table"][hier["pseudoF_table"]["k"] == best_k].iloc[0]
            plt.scatter([best_k], [best_row["pseudoF"]], s=100)
            plt.annotate(
                f"opt k={best_k}",
                (best_k, best_row["pseudoF"]),
                xytext=(best_k + 0.3, best_row["pseudoF"]),
            )
            plt.title(f"{run_name}: pseudo-F для k = 2..20")
            plt.xlabel("Число кластеров k")
            plt.ylabel("pseudo-F (Calinski-Harabasz)")
            plt.grid(alpha=0.3)
            plt.show()

            # Проекция для иерархических меток
            plot_projection(
                projection_coords,
                hier_labels,
                f"{run_name}: проекция {projection_label} с метками иерархической кластеризации",
            )

            # Проекция для прототипной кластеризации
            plot_projection(
                projection_coords,
                proto_labels,
                f"{run_name}: проекция {projection_label} с метками {proto['method_label']}",
            )

        return RunResult(
            name=run_name,
            X=X,
            feature_names=prepared["feature_names"],
            hier_labels=hier_labels,
            proto_labels=proto_labels,
            best_k=best_k,
            pseudoF_table=hier["pseudoF_table"],
            projection_coords=projection_coords,
            projection_label=projection_label,
            representatives=representatives,
            skewed_columns=prepared["skewed_columns"],
            selected_numeric_features=selected_numeric_features,
        )

    def select_features_varclus(self):
        """
        Для VarClus берём числовые признаки после импутации, но до one-hot и до масштабирования.
        Это логично, потому что мы хотим отбирать сами переменные, а не уже разложенные дамми-признаки.
        """
        prepared = self._prepare_data(skew_transform=False, selected_numeric_features=None)
        numeric_imputed = prepared["feature_names"]  # просто чтобы показать, что prepare_data уже отработал

        df = self.df_raw.copy()
        _, numeric_cols, _ = get_column_groups(df)
        imputer = build_imputer(self.config["imputer"])
        numeric_imputed_df = pd.DataFrame(
            imputer.fit_transform(df[numeric_cols]),
            columns=numeric_cols,
            index=df.index,
        )
        numeric_imputed_df["logSalary"] = np.log1p(numeric_imputed_df["Salary"])

        selected, clusters, variable_Z = varclus_like_selection(
            numeric_imputed_df,
            n_clusters=self.config["varclus_clusters"],
        )

        return selected, clusters, variable_Z

## 5. Создаём объект анализа

Теперь инициализируем класс и загружаем данные.  
После этого все основные действия можно вызывать через методы объекта `study`.

In [ ]:
study = BaseballClusterStudy(variant=ACTIVE_VARIANT, random_state=RANDOM_STATE)
study.load_data(DATA_PATH)

print("Текущая конфигурация варианта:")
display(pd.Series(study.config))

## 6. Базовый запуск — реализация шагов 1–8

Сейчас запускаем основной конвейер без дополнительной предобработки из пункта 9 и без отбора признаков из пункта 10.

### Что здесь происходит по шагам
- **Шаг 2**: импутация `Salary`.
- **Шаг 2**: пересчёт `logSalary = log(1 + Salary)`.
- **Шаг 3**: масштабирование числовых и one-hot кодирование категориальных.
- **Шаг 4**: агломеративная иерархическая кластеризация.
- **Шаг 5**: pseudo-F для `k = 2..20` и выбор первого локального пика.
- **Шаг 6**: 2D-проекция.
- **Шаг 7**: прототипная кластеризация.
- **Шаг 8**: всё это уже собрано в класс.

In [ ]:
base_result = study.run(
    run_name="Базовый запуск",
    skew_transform=False,
    selected_numeric_features=None,
    show_plots=True,
)

### Что стоит посмотреть после базового запуска

1. **Дендрограмма**  
   Показывает, как кластеры постепенно склеиваются.  
   Чем выше вертикальная линия склейки, тем более далекими были объединённые группы.

2. **График pseudo-F**  
   Мы рассматриваем все варианты от 2 до 20 кластеров.  
   В качестве оптимума берём **первый локальный максимум**, потому что это прямо указано в задании.

3. **Проекция на плоскость**  
   Она не создаёт кластеры сама по себе, а только помогает визуально понять структуру данных.

4. **Прототипная кластеризация**  
   Здесь уже используется метод из варианта: `KMeans`, `KMedoids` или `EM`.

5. **Типичные представители**  
   Это реальные игроки, которые ближе всего к центру/медоиду своего кластера.

In [ ]:
print("Оптимальное число кластеров (по первому локальному пику pseudo-F):", base_result.best_k)
print("Использованный метод проекции:", base_result.projection_label)
print_cluster_sizes(base_result.hier_labels, "Размеры кластеров (иерархическая кластеризация)")
print_cluster_sizes(base_result.proto_labels, "Размеры кластеров (прототипная кластеризация)")
display(base_result.representatives)

## 7. Интерпретация базового результата

Здесь удобно зафиксировать краткий вывод словами.

### На что ориентироваться в тексте отчёта
- сколько кластеров получилось;
- насколько они разделяются на проекции;
- похож ли результат иерархической и прототипной кластеризации;
- есть ли маленькие или неустойчивые кластеры;
- насколько осмысленно выглядят типичные представители.

Ниже я вывожу короткую автоматическую сводку, которую потом можно использовать как основу для текста в отчёте.

In [ ]:
base_summary = pd.DataFrame({
    "run": [base_result.name],
    "best_k": [base_result.best_k],
    "projection": [base_result.projection_label],
    "n_features_after_encoding": [len(base_result.feature_names)],
})
display(base_summary)

print("Типичные представители по кластерам:")
for _, row in base_result.representatives.iterrows():
    print(
        f"Кластер {int(row['cluster'])}: "
        f"{row['Name']} (размер кластера = {int(row['cluster_size'])})"
    )

## 8. Шаг 9 — дополнительная предобработка через `log(1 + x)` для асимметричных признаков

По условию задания надо:
1. найти признаки с одной модой и тяжёлым правым хвостом;
2. применить к ним `log(1 + x)`;
3. заново запустить весь конвейер из шага 8.

### Очень важный смысловой момент

Логарифмирование нужно делать **до масштабирования**, а не после него.

Почему:
- после `StandardScaler`, `RobustScaler` и некоторых других преобразований значения могут стать отрицательными;
- для отрицательных значений `log(1 + x)` уже может быть не определён;
- значит логарифмирование нужно применять именно к **сырым числовым признакам после импутации**, но до нормализации.

В этом ноутбуке именно так и сделано.

In [ ]:
log_result = study.run(
    run_name="После дополнительного log(1+x)",
    skew_transform=True,
    selected_numeric_features=None,
    show_plots=True,
)

print("Логарифмированы признаки:")
print(log_result.skewed_columns if log_result.skewed_columns else "Подходящих признаков по выбранному правилу не нашлось.")

print("Оптимальное число кластеров после логарифмирования:", log_result.best_k)
display(log_result.representatives)

### Как интерпретировать шаг 9

После логарифмирования распределения правосторонне асимметричных признаков часто становятся более компактными:
- уменьшается влияние очень больших значений;
- крупные выбросы меньше «тянут» расстояния на себя;
- кластеры иногда становятся визуально более плотными.

Но это не гарантирует автоматически «лучший» результат:
- число кластеров может измениться;
- проекция может стать и лучше, и хуже;
- типичные представители могут смениться.

Поэтому ниже строим прямое сравнение базового запуска и запуска после `log(1 + x)`.

In [ ]:
comparison_step9 = pd.DataFrame({
    "run": [base_result.name, log_result.name],
    "best_k": [base_result.best_k, log_result.best_k],
    "projection": [base_result.projection_label, log_result.projection_label],
    "n_features_after_encoding": [len(base_result.feature_names), len(log_result.feature_names)],
    "transformed_columns_count": [0, len(log_result.skewed_columns)],
})
display(comparison_step9)

## 9. Шаг 10 — отбор признаков методом VarClus

Теперь нужно сократить число признаков, выбрав наиболее значимые.

### Что такое VarClus по смыслу

Вместо кластеризации **объектов** мы кластеризуем **переменные**:
- если две переменные очень похожи, значит они несут близкую информацию;
- хранить обе не всегда полезно;
- можно выбрать одного представителя группы похожих признаков.

### Как это реализовано здесь

Так как в стандартном `scikit-learn` нет прямой реализации SAS VarClus, я использую очень близкую по смыслу схему:
1. беру числовые признаки после импутации;
2. считаю модуль корреляции между переменными;
3. кластеризую переменные;
4. в каждом кластере выбираю признак-представитель.

Это хорошая, понятная и воспроизводимая учебная реализация.

In [ ]:
selected_features, varclus_clusters, variable_Z = study.select_features_varclus()

print("Отобранные числовые признаки:")
print(selected_features)

print("\nКластеры признаков:")
for cluster_id, features in varclus_clusters.items():
    print(f"Кластер переменных {cluster_id}: {features}")

### Дендрограмма переменных

Эта дендрограмма показывает, какие **признаки** похожи друг на друга.
Если несколько переменных склеиваются рано, значит они содержат близкую информацию.

In [ ]:
plt.figure(figsize=(12, 6))
dendrogram(
    variable_Z,
    labels=list(study.df_raw.drop(columns=["Name"]).select_dtypes(include=[np.number]).columns) + []
)
plt.title("Дендрограмма переменных (VarClus-подобный шаг)")
plt.xlabel("Признаки")
plt.ylabel("Расстояние между признаками")
plt.xticks(rotation=90)
plt.grid(alpha=0.3)
plt.show()

Теперь снова запускаем основной конвейер, но уже не на полном наборе числовых признаков, а только на выбранных представителях кластеров признаков.

In [ ]:
varclus_result = study.run(
    run_name="После отбора признаков VarClus",
    skew_transform=False,
    selected_numeric_features=selected_features,
    show_plots=True,
)

print("Оптимальное число кластеров после VarClus:", varclus_result.best_k)
display(varclus_result.representatives)

## 10. Сравнение всех трёх запусков

Теперь у нас есть три сценария:
1. базовый;
2. после дополнительного логарифмирования;
3. после отбора признаков.

Их удобно сравнить в одной таблице.

In [ ]:
final_comparison = pd.DataFrame({
    "run": [base_result.name, log_result.name, varclus_result.name],
    "best_k": [base_result.best_k, log_result.best_k, varclus_result.best_k],
    "projection": [base_result.projection_label, log_result.projection_label, varclus_result.projection_label],
    "n_features_after_encoding": [
        len(base_result.feature_names),
        len(log_result.feature_names),
        len(varclus_result.feature_names),
    ],
    "selected_numeric_features": [
        None,
        None,
        ", ".join(selected_features),
    ],
    "skewed_columns": [
        None,
        ", ".join(log_result.skewed_columns) if log_result.skewed_columns else None,
        None,
    ],
})
display(final_comparison)

## 11. Готовые заготовки выводов для отчёта

Ниже — не «истина в последней инстанции», а удобная болванка, которую можно почти сразу вставлять в текст работы и затем слегка править под реальные картинки.

Важно: сначала **запусти ноутбук**, посмотри на графики, а потом проверь, что автоматический текст не противоречит тому, что ты реально видишь.

In [ ]:
print("=== Краткий текст по шагу 5 ===")
print(
    f"По критерию pseudo-F оптимальное число кластеров для базового запуска "
    f"было выбрано равным {base_result.best_k}, поскольку это первый локальный пик "
    f"на графике зависимости критерия от числа кластеров."
)

print("\n=== Краткий текст по шагу 9 ===")
print(
    f"После дополнительного преобразования log(1+x) число кластеров изменилось "
    f"с {base_result.best_k} до {log_result.best_k}. "
    f"Субъективное качество кластеризации следует оценивать по тому, "
    f"стали ли группы визуально компактнее и уменьшилось ли влияние выбросов "
    f"на проекцию."
)

print("\n=== Краткий текст по шагу 10 ===")
print(
    f"После отбора признаков методом VarClus-подобной кластеризации переменных "
    f"число кластеров стало {varclus_result.best_k}. "
    f"Отбор признаков уменьшает избыточность описания объектов и может сделать "
    f"структуру кластеров более интерпретируемой."
)

## 12. Где именно в ноутбуке закрыты пункты задания

Чтобы тебе было проще ориентироваться:

- **Пункт 2** — импутация и пересчёт `logSalary` в методе `_prepare_data`.
- **Пункт 3** — масштабирование и `OneHotEncoder` там же.
- **Пункт 4** — `_hierarchical_stage`, плюс график дендрограммы.
- **Пункт 5** — таблица и график `pseudo-F`, выбор первого локального пика.
- **Пункт 6** — `make_projection`.
- **Пункт 7** — `prototype_clustering` и `typical_representatives`.
- **Пункт 8** — класс `BaseballClusterStudy`.
- **Пункт 9** — запуск `log_result`.
- **Пункт 10** — `select_features_varclus()` и запуск `varclus_result`.

---

## Последняя практическая рекомендация

Перед сдачей:
1. проверь `ACTIVE_VARIANT`;
2. выполни ноутбук сверху вниз;
3. посмотри, чтобы все графики построились;
4. перечитай текстовые выводы и поправь формулировки под свои реальные результаты.

Этот ноутбук написан так, чтобы его можно было не просто сдать, а ещё и **понять**.